In [2]:

# TASK 19B: VERIFY REQUIRED LIBRARIES

import mysql.connector
import sqlalchemy
import pandas as pd
from dotenv import load_dotenv

print("All required libraries imported successfully!")


All required libraries imported successfully!


In [3]:

# TASK 19B: CONNECT PYTHON TO MYSQL

import os
from pathlib import Path
from dotenv import load_dotenv
import mysql.connector

# Step 1: Locate our project folder
project_folder = Path(
    r"C:\Users\shree\Desktop\KYC_Customer_Risk_Assessment"
)

# Step 2: Load MySQL settings from .env
load_dotenv(project_folder / ".env")

# Step 3: Connect to MySQL
conn = mysql.connector.connect(
    host=os.getenv("MYSQL_HOST"),
    port=int(os.getenv("MYSQL_PORT", "3306")),
    user=os.getenv("MYSQL_USER"),
    password=os.getenv("MYSQL_PASSWORD"),
    database=os.getenv("MYSQL_DATABASE")
)

# Step 4: Test database connection
try:
    cursor = conn.cursor()

    cursor.execute("SELECT COUNT(*) FROM kyc_cases")

    total_cases = cursor.fetchone()[0]

    print("MySQL connected successfully!")
    print("Database:", conn.database)
    print("Total KYC cases:", total_cases)

finally:
    cursor.close()
    conn.close()


MySQL connected successfully!
Database: clearcase_kyc
Total KYC cases: 1630


In [5]:

# TASK 19C: RETRIEVE MYSQL DATA USING PANDAS

import os
import pandas as pd

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Create MySQL connection settings
db_url = URL.create(
    drivername="mysql+mysqlconnector",
    username=os.getenv("MYSQL_USER"),
    password=os.getenv("MYSQL_PASSWORD"),
    host=os.getenv("MYSQL_HOST"),
    port=int(os.getenv("MYSQL_PORT", "3306")),
    database=os.getenv("MYSQL_DATABASE")
)

# Create SQLAlchemy engine
engine = create_engine(db_url)

# Retrieve 10 KYC cases
query = text("""
    SELECT
        case_id,
        customer_id,
        review_type,
        case_status,
        priority
    FROM kyc_cases
    LIMIT 10
""")

# Execute query and create DataFrame
with engine.connect() as connection:
    df = pd.read_sql(query, connection)

# Display results
display(df)


,case_id,customer_id,review_type,case_status,priority
0,KYC000001,CUS00001,ONBOARDING,ESCALATED,HIGH
1,KYC000002,CUS00002,ONBOARDING,IN_REVIEW,HIGH
2,KYC000003,CUS00003,ONBOARDING,PENDING_SCREENING,HIGH
3,KYC000004,CUS00004,ONBOARDING,COMPLETED,HIGH
4,KYC000005,CUS00005,ONBOARDING,COMPLETED,HIGH
5,KYC000006,CUS00006,ONBOARDING,COMPLETED,HIGH
6,KYC000007,CUS00007,ONBOARDING,COMPLETED,HIGH
7,KYC000008,CUS00008,ONBOARDING,ESCALATED,HIGH
8,KYC000009,CUS00009,ONBOARDING,COMPLETED,HIGH
9,KYC000010,CUS00010,ONBOARDING,COMPLETED,HIGH


In [6]:

# TASK 19D: VALIDATE MYSQL DATA IN PYTHON

# Count all KYC cases by status
query = text("""
    SELECT
        case_status,
        COUNT(*) AS total_cases
    FROM kyc_cases
    GROUP BY case_status
    ORDER BY total_cases DESC
""")

# Retrieve results using SQLAlchemy
with engine.connect() as connection:
    status_df = pd.read_sql(query, connection)

# Display case counts by status
display(status_df)

# Calculate total KYC cases
total_cases = status_df["total_cases"].sum()

# Calculate completed KYC cases
completed_cases = status_df.loc[
    status_df["case_status"] == "COMPLETED",
    "total_cases"
].sum()

print("Total KYC cases:", total_cases)
print("Completed KYC cases:", completed_cases)


,case_status,total_cases
0,COMPLETED,1259
1,IN_REVIEW,88
2,PENDING_INFORMATION,83
3,ESCALATED,70
4,PENDING_SCREENING,45
5,QC_REWORK,37
6,AWAITING_QC,31
7,DECLINED,17


Total KYC cases: 1630
Completed KYC cases: 1259
